# 02 — Data Quality Profiling (RQ1)
*How reliable is the available energy data?*

Produces the data-quality scorecard: missingness, duplicate timestamps, gaps,
zero readings, unrealistic peaks, and metadata completeness.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import plotly.express as px

from src.data.ingestion import load_dataset
from src.data.cleaning import clean_dataset

# Default: synthetic data (offline). Switch to source="bdg2" for real BDG2 data.
ds = load_dataset(source="synthetic", n_buildings=8)
print(f"{ds.meters.shape[0]:,} readings | {ds.metadata.shape[0]} buildings")
ds.metadata

## Scorecard on the raw data

In [ ]:
from src.data.quality import profile_dataset
scorecard = profile_dataset(ds.meters, ds.metadata)
scorecard.style.background_gradient(subset=["quality_score"], cmap="RdYlGn", vmin=50, vmax=100)

## Cleaning and its audit trail
Every repair action is counted so the report can document the cleaning choices.

In [ ]:
clean, report = clean_dataset(ds.meters)
report.to_frame()

In [ ]:
print("Totals across all buildings:")
report.totals()

## Before/after visual check around a repaired gap

In [ ]:
b = clean["building_id"].iloc[0]
imp = clean[(clean["building_id"] == b) & clean["imputed"]]
if len(imp):
    t0 = imp["timestamp"].iloc[0]
    window = clean[(clean["building_id"] == b)
                   & clean["timestamp"].between(t0 - pd.Timedelta("2D"), t0 + pd.Timedelta("2D"))]
    fig = px.line(window, x="timestamp", y="meter_reading", title=f"Repaired gap — {b}")
    fig.add_scatter(x=imp["timestamp"], y=imp["meter_reading"], mode="markers",
                    name="imputed", marker=dict(color="red", size=7))
    fig.show()

## Conclusions (feed into the technical report)
- Quality score per building quantifies RQ1; issues found: duplicates, negatives,
  short/long gaps, extreme peaks.
- Cleaning is conservative: only physically impossible values and short gaps are
  repaired; suspicious-but-possible readings are left for anomaly detection.